# [Preliminary] Chronos embeddings — time-based split

> **Preliminary only.** Final results are in `output/models_table.txt` (produced by `train/train_table_iii_cloud.py`). Do not cite numbers from this notebook.

Frozen [Chronos](https://github.com/amazon-science/chronos-forecasting) encoder applied to 75-day, 15-feature GRIDMET sequences (one univariate pass per feature, mean-pooled and concatenated). Three classification heads trained on the resulting embeddings.

**Split:** time-based — train before 2025-01-01, test Jan–Apr 2025. Test set is ~614 sequences; metrics should be interpreted cautiously.

> `pip install chronos-forecasting xgboost`

In [1]:
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from chronos import ChronosPipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

/dstack/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [2]:
DATA_PATH = "Wildfire_Dataset.csv"
SEQ_LEN = 75
FILL_VALUE = 32767.0

FEATURES = [
    "pr", "rmax", "rmin", "sph", "srad", "tmmn", "tmmx", "vs",
    "bi", "fm100", "fm1000", "erc", "etr", "pet", "vpd",
]

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])
df = df.loc[~(df == FILL_VALUE).any(axis=1)].copy()
df = df.sort_values(["latitude", "longitude", "datetime"]).reset_index(drop=True)

df["seq_id"] = np.arange(len(df)) // SEQ_LEN
groups = [g for _, g in df.groupby("seq_id") if len(g) == SEQ_LEN]

seqs = np.stack([g[FEATURES].to_numpy(dtype=np.float32) for g in groups])
labels = np.array([int((g["Wildfire"] == "Yes").any()) for g in groups], dtype=np.int64)
# Last datetime in each window — used for time-based splitting below.
anchor_dates = np.array([g["datetime"].max() for g in groups])

print(f"Sequences: {seqs.shape}  |  positive share: {labels.mean():.3f}")

Sequences: (126456, 75, 15)  |  positive share: 0.263


In [3]:
# ── Split options ─────────────────────────────────────────────────────────────
# TIME_SPLIT = True   → paper-faithful: train on all data before TEST_START,
#                        test on Jan–Apr 2025 (matches the paper's held-out period).
# TIME_SPLIT = False  → random 80/20 stratified split (faster to iterate on).
#
# MAX_SEQUENCES only applies when TIME_SPLIT = False (subsampling a random split).
# Set to None for the full dataset run.
TIME_SPLIT = True
TEST_START = pd.Timestamp("2025-01-01")
MAX_SEQUENCES = 5000  # ignored when TIME_SPLIT = True

if TIME_SPLIT:
    test_mask = anchor_dates >= TEST_START
    train_mask = ~test_mask
    X_train, X_test = seqs[train_mask], seqs[test_mask]
    y_train, y_test = labels[train_mask], labels[test_mask]
else:
    if MAX_SEQUENCES and len(seqs) > MAX_SEQUENCES:
        idx = np.random.default_rng(SEED).permutation(len(seqs))[:MAX_SEQUENCES]
        seqs, labels = seqs[idx], labels[idx]
    X_train, X_test, y_train, y_test = train_test_split(
        seqs, labels, test_size=0.20, stratify=labels, random_state=SEED
    )

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Train positives: {y_train.sum()}/{len(y_train)}  Test positives: {y_test.sum()}/{len(y_test)}")
print(f"Split mode: {'time-based (paper-faithful)' if TIME_SPLIT else 'random stratified'}")

Train: (125842, 75, 15)  Test: (614, 75, 15)
Train positives: 32797/125842  Test positives: 488/614
Split mode: time-based (paper-faithful)


In [4]:
CHRONOS_MODEL = "amazon/chronos-t5-mini"  # swap to t5-tiny for faster runs
EMBED_BATCH = 256

pipeline = ChronosPipeline.from_pretrained(
    CHRONOS_MODEL, device_map=str(device), torch_dtype=torch.float32
)


@torch.inference_mode()
def chronos_features(
    seq_array: np.ndarray,
    batch_size: int = EMBED_BATCH,
    feature_names: list | None = None,
) -> np.ndarray:
    """Return (N, F * d_model) embeddings from a frozen Chronos encoder.

    Each univariate (sample, feature) slice is mean-pooled over the context
    dimension; the F per-feature embeddings are concatenated.
    """
    n, _, n_features = seq_array.shape
    names = feature_names or FEATURES[:n_features]
    blocks = []
    for f in range(n_features):
        slices = []
        for start in range(0, n, batch_size):
            ctx = torch.tensor(seq_array[start:start + batch_size, :, f])
            emb, _ = pipeline.embed(ctx)
            slices.append(emb.mean(dim=1).cpu().numpy())
        blocks.append(np.concatenate(slices))
        print(f"  {f + 1:02d}/{n_features} {names[f]}")
    return np.concatenate(blocks, axis=1)


EMB_DIR = Path("data/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

split_tag = "timesplit" if TIME_SPLIT else f"random{MAX_SEQUENCES or 'full'}"
model_tag = CHRONOS_MODEL.replace("/", "_")
train_path = EMB_DIR / f"{model_tag}_{split_tag}_train.npy"
test_path  = EMB_DIR / f"{model_tag}_{split_tag}_test.npy"

if train_path.exists() and test_path.exists():
    X_train_emb = np.load(train_path)
    X_test_emb = np.load(test_path)
    print(f"Loaded cached embeddings from {EMB_DIR}")
else:
    print("Encoding train...")
    X_train_emb = chronos_features(X_train)
    print("Encoding test...")
    X_test_emb = chronos_features(X_test)
    np.save(train_path, X_train_emb)
    np.save(test_path, X_test_emb)
    print(f"Saved embeddings to {EMB_DIR}")

print("Embedding shape:", X_train_emb.shape)

`torch_dtype` is deprecated! Use `dtype` instead!


Encoding train...


  01/15 pr


  02/15 rmax


  03/15 rmin


...

  11/15 fm1000


  12/15 erc
  13/15 etr


  14/15 pet


  15/15 vpd


Saved embeddings to data/embeddings
Embedding shape: (125842, 5760)


In [5]:
# Performance improvement: PCA reduces 5760-dim embeddings to 256 dims.
# This removes noise, cuts head training time ~20x, and avoids memory pressure on MPS.
PCA_N = 256

pca = PCA(n_components=PCA_N, random_state=SEED).fit(X_train_emb)
X_train_pca = pca.transform(X_train_emb)
X_test_pca = pca.transform(X_test_emb)
print(f"Variance explained by {PCA_N} components: {pca.explained_variance_ratio_.sum():.3f}")

scaler = StandardScaler().fit(X_train_pca)
X_train_s = scaler.transform(X_train_pca)
X_test_s = scaler.transform(X_test_pca)

Variance explained by 256 components: 0.801


In [6]:
def metric_row(name, y_true, y_pred, probs=None):
    """Return a dict of metrics. Pass probs to include ROC-AUC and PR-AUC."""
    row = {
        "Model": name,
        "Accuracy [%]": 100 * accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "F2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
    }
    if probs is not None:
        row["ROC-AUC"] = roc_auc_score(y_true, probs)
        row["PR-AUC"] = average_precision_score(y_true, probs)
    return row


def best_threshold_row(name, y_true, probs):
    """Re-threshold at the F1-maximising cutpoint; include all prob-based metrics."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, probs)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best = float(thresholds[f1s[:-1].argmax()])
    return metric_row(f"{name} (τ={best:.2f})", y_true, (probs >= best).astype(int), probs=probs)


def display_table(rows):
    """Format results as a sorted DataFrame."""
    return (
        pd.DataFrame(rows)
        .sort_values("Accuracy [%]", ascending=False)
        .reset_index(drop=True)
        .round({"Accuracy [%]": 1, "Precision": 2, "Recall": 2, "F1": 2, "F2": 2, "ROC-AUC": 3, "PR-AUC": 3})
    )


results = []

In [7]:
logreg = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=SEED)
logreg.fit(X_train_s, y_train)

lr_probs = logreg.predict_proba(X_test_s)[:, 1]
results.append(metric_row("Chronos + LR", y_test, logreg.predict(X_test_s), probs=lr_probs))
results.append(best_threshold_row("Chronos + LR", y_test, lr_probs))
display_table(results)

,Model,Accuracy [%],Precision,Recall,F1,F2,ROC-AUC,PR-AUC
0,Chronos + LR (τ=0.38),90.4,0.92,0.97,0.94,0.96,0.952,0.982
1,Chronos + LR,88.3,0.96,0.89,0.92,0.90,0.952,0.982


In [8]:
class MLPHead(nn.Module):
    """Two-hidden-layer MLP with BatchNorm for stable training on PCA embeddings."""

    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(256, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


def make_loader(X, y, batch_size=128, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_mlp(model, loader, epochs=8, lr=1e-3):
    model = model.to(device)
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        loss_sum = 0.0
        for Xb, yb in loader:
            opt.zero_grad()
            loss = criterion(model(Xb.to(device)), yb.to(device))
            loss.backward()
            opt.step()
            loss_sum += loss.item()
        print(f"Epoch {epoch + 1}/{epochs}  loss={loss_sum / len(loader):.4f}")
    return model


@torch.no_grad()
def mlp_proba(model, X):
    """Return positive-class softmax probabilities for array X."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32)
    return torch.cat([
        torch.softmax(model(X_t[i:i + 256].to(device)), dim=1)[:, 1]
        for i in range(0, len(X_t), 256)
    ]).cpu().numpy()


mlp = train_mlp(MLPHead(X_train_s.shape[1]), make_loader(X_train_s, y_train, shuffle=True))
mlp_probs = mlp_proba(mlp, X_test_s)
results.append(metric_row("Chronos + MLP", y_test, (mlp_probs >= 0.5).astype(int), probs=mlp_probs))
results.append(best_threshold_row("Chronos + MLP", y_test, mlp_probs))
display_table(results)

Epoch 1/8  loss=0.6592


Epoch 2/8  loss=0.6332


Epoch 3/8  loss=0.6150


Epoch 4/8  loss=0.5983


Epoch 5/8  loss=0.5834


Epoch 6/8  loss=0.5694


Epoch 7/8  loss=0.5559


Epoch 8/8  loss=0.5443


,Model,Accuracy [%],Precision,Recall,F1,F2,ROC-AUC,PR-AUC
0,Chronos + LR (τ=0.38),90.4,0.92,0.97,0.94,0.96,0.952,0.982
1,Chronos + MLP (τ=0.29),89.3,0.92,0.94,0.93,0.94,0.932,0.981
2,Chronos + LR,88.3,0.96,0.89,0.92,0.90,0.952,0.982
3,Chronos + MLP,80.5,0.96,0.78,0.86,0.81,0.932,0.981


In [9]:
if XGBOOST_AVAILABLE:
    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
        n_jobs=-1, random_state=SEED,
    )
    xgb.fit(X_train_s, y_train)
    xgb_probs = xgb.predict_proba(X_test_s)[:, 1]
    results.append(metric_row("Chronos + XGBoost", y_test, xgb.predict(X_test_s), probs=xgb_probs))
    results.append(best_threshold_row("Chronos + XGBoost", y_test, xgb_probs))
else:
    print("XGBoost not installed — skipping. Run: pip install xgboost")

display_table(results)

,Model,Accuracy [%],Precision,Recall,F1,F2,ROC-AUC,PR-AUC
0,Chronos + XGBoost (τ=0.18),92.0,0.94,0.97,0.95,0.96,0.958,0.988
1,Chronos + LR (τ=0.38),90.4,0.92,0.97,0.94,0.96,0.952,0.982
2,Chronos + MLP (τ=0.29),89.3,0.92,0.94,0.93,0.94,0.932,0.981
3,Chronos + LR,88.3,0.96,0.89,0.92,0.90,0.952,0.982
4,Chronos + MLP,80.5,0.96,0.78,0.86,0.81,0.932,0.981
5,Chronos + XGBoost,57.5,1.00,0.47,0.64,0.52,0.958,0.988


In [10]:
# Performance improvement: encode only the 5 fire-relevant GRIDMET variables.
# Removing weather noise (precipitation, humidity, etc.) often sharpens the signal
# for fire-index-driven ignition patterns.
FIRE_FEATURES = ["erc", "bi", "vpd", "fm100", "fm1000"]
fire_col_idx = [FEATURES.index(f) for f in FIRE_FEATURES]

print("Encoding fire-index subset (train)...")
X_train_fire_emb = chronos_features(X_train[:, :, fire_col_idx], feature_names=FIRE_FEATURES)
print("Encoding fire-index subset (test)...")
X_test_fire_emb = chronos_features(X_test[:, :, fire_col_idx], feature_names=FIRE_FEATURES)

fire_pca = PCA(n_components=min(PCA_N, X_train_fire_emb.shape[1]), random_state=SEED).fit(X_train_fire_emb)
X_train_fire_pca = fire_pca.transform(X_train_fire_emb)
fire_scaler = StandardScaler().fit(X_train_fire_pca)
X_train_fire_s = fire_scaler.transform(X_train_fire_pca)
X_test_fire_s = fire_scaler.transform(fire_pca.transform(X_test_fire_emb))
print(f"Fire-index variance explained: {fire_pca.explained_variance_ratio_.sum():.3f}")

fire_lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=SEED)
fire_lr.fit(X_train_fire_s, y_train)
fire_probs = fire_lr.predict_proba(X_test_fire_s)[:, 1]
results.append(metric_row("Chronos (fire idx) + LR", y_test, fire_lr.predict(X_test_fire_s), probs=fire_probs))
results.append(best_threshold_row("Chronos (fire idx) + LR", y_test, fire_probs))
display_table(results)

Encoding fire-index subset (train)...


  01/5 erc


  02/5 bi


  03/5 vpd


  04/5 fm100


...

  02/5 bi


  03/5 vpd


  04/5 fm100
  05/5 fm1000


Fire-index variance explained: 0.915


,Model,Accuracy [%],Precision,Recall,F1,F2,ROC-AUC,PR-AUC
0,Chronos + XGBoost (τ=0.18),92.0,0.94,0.97,0.95,0.96,0.958,0.988
1,Chronos + LR (τ=0.38),90.4,0.92,0.97,0.94,0.96,0.952,0.982
2,Chronos (fire idx) + LR (τ=0.37),89.7,0.91,0.97,0.94,0.96,0.914,0.969
3,Chronos + MLP (τ=0.29),89.3,0.92,0.94,0.93,0.94,0.932,0.981
4,Chronos + LR,88.3,0.96,0.89,0.92,0.90,0.952,0.982
5,Chronos (fire idx) + LR,84.0,0.95,0.84,0.89,0.86,0.914,0.969
6,Chronos + MLP,80.5,0.96,0.78,0.86,0.81,0.932,0.981
7,Chronos + XGBoost,57.5,1.00,0.47,0.64,0.52,0.958,0.988


In [11]:
chronos_df = display_table(results)

try:
    paper_df = pd.read_csv("table_iii_model_comparison.csv")
    combined = (
        pd.concat([paper_df, chronos_df], ignore_index=True)
        .sort_values("Accuracy [%]", ascending=False)
        .reset_index(drop=True)
        .round({"Accuracy [%]": 1, "Precision": 2, "Recall": 2, "F1": 2})
    )
    display(combined)
    combined.to_csv("table_chronos_vs_paper.csv", index=False)
    print("Wrote table_chronos_vs_paper.csv")
except FileNotFoundError:
    print("table_iii_model_comparison.csv not found — run replicating_table.ipynb first.")
    display(chronos_df)

table_iii_model_comparison.csv not found — run replicating_table.ipynb first.


,Model,Accuracy [%],Precision,Recall,F1,F2,ROC-AUC,PR-AUC
0,Chronos + XGBoost (τ=0.18),92.0,0.94,0.97,0.95,0.96,0.958,0.988
1,Chronos + LR (τ=0.38),90.4,0.92,0.97,0.94,0.96,0.952,0.982
2,Chronos (fire idx) + LR (τ=0.37),89.7,0.91,0.97,0.94,0.96,0.914,0.969
3,Chronos + MLP (τ=0.29),89.3,0.92,0.94,0.93,0.94,0.932,0.981
4,Chronos + LR,88.3,0.96,0.89,0.92,0.90,0.952,0.982
5,Chronos (fire idx) + LR,84.0,0.95,0.84,0.89,0.86,0.914,0.969
6,Chronos + MLP,80.5,0.96,0.78,0.86,0.81,0.932,0.981
7,Chronos + XGBoost,57.5,1.00,0.47,0.64,0.52,0.958,0.988
